# 🎯 [PRC07_M02] 국내외 공급업체 데이터 정제, 부품 단종(DMSMS) 공고 파싱 전처리 모듈 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m02_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M02 |
| **미션 주제** | 국내외 공급업체 데이터 정제, 부품 단종(DMSMS) 공고 파싱 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip() + "\n")
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 02. 국내외 공급업체 데이터 정제, 부품 단종(DMSMS) 공고 파싱 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

공급업체 데이터 정제 및 부품 단종 공고 파싱 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **국내외 공급업체 자료의 표기 차이를 정제하고 서식이 다른 부품 단종(DMSMS) 공고문에서 필요한 항목을 추출하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **CSV, 엑셀 내보내기 형태(탭 구분), JSON 세 형식의 공급업체 자료를 열 이름 별칭 사전으로 하나의 표준 스키마로 바꾸고, 업체명 표기와 국가 코드, 리드타임 단위(일·주·개월·년), 날짜 형식을 통일 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/connectors/prc07_data_loader.py 및 스키마 정의서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 국방 서식/양식 변경에 대한 유연한 확장성

#### 미션 PASS 조건

- 정량 기준: 지원 파일 포맷(CSV, Excel, JSON 등) 파싱 성공률 100%


- 진행 기준: 데이터 로더 및 스키마 유효성 검사 모듈 완성


- 검증 방법: 다양한 형태의 샘플 군수 문서 대상 파싱 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""국내외 공급업체 자료(CSV·엑셀 내보내기 TSV·JSON)를 하나의 표준 스키마로 정제하고, 문서마다 서식이 다른 부품 단종(DMSMS) 공고문 텍스트에서 부품번호·단종일·최종구매(LTB) 기한·대체품 정보를 파싱하는 전처리 모듈을 구현한다."""
import csv
import io
import json
import re
from datetime import date
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 서로 다른 형식의 원시 자료 (실제 업체·공고 아님). 표기가 일부러 제각각이다.
# ---------------------------------------------------------------------------
RAW_CSV = """업체코드,업체명,국가,공급부품,리드타임,최근거래일
V001,(주)가상정밀,국내,P-0001,45일,2026.03.02
V002, 가상전자 주식회사 ,KR,P-0004,2개월,2026-3-15
V003,Virtual Parts Inc.,A국,P-0003,180,20260110
V004,가상유공압㈜,국내,P-0003,60일,26/02/27
V005,VIRTUAL PARTS INC,가상국가A,P-0011,6개월,2025.12.30
V006,Sample Optics GmbH,B,P-0005,210일,2026-01-05
"""
RAW_TSV = "공급업체\t국적\t품목번호\t납기(일)\t최종거래\n" \
          "가상광학(주)\t대한민국\tP-0005\t75\t2026.02.11\n" \
          "Sample Optics GmbH\t가상국가B\tP-0013\t200\t2026.01.20\n" \
          "가상테크 (주)\t국내\tP-0008\t90\t2026/03/05\n" \
          "Model Sensor Co.\tC국\tP-0006\t1년\t2025.11.03\n" \
          "Virtual Parts Inc\tA국\tP-0003\t180\t2026.01.10\n"
RAW_JSON = json.dumps({"result": "OK", "items": [
    {"vendor": {"name": "가상시스템(주)", "nation": "KR"}, "part": "P-0016", "leadTime": {"value": 3, "unit": "month"}, "lastTrade": "2026-02-01"},
    {"vendor": {"name": "Model Sensor Co", "nation": "C"}, "part": "P-0014", "leadTime": {"value": 150, "unit": "day"}, "lastTrade": "2026-01-09"},
    {"vendor": {"name": "가상정밀 주식회사", "nation": "KR"}, "part": "P-0002", "leadTime": {"value": 8, "unit": "week"}, "lastTrade": "2026-03-20"}]},
    ensure_ascii=False)
NOTICES = [
    "제목: 부품 단종 예정 통보\n부품번호: P-0003\n단종 예정일: 2027년 3월 31일\n최종구매(LTB) 마감: 2026년 12월 31일\n대체품: 없음",
    "[DMSMS NOTICE] Part No. P-0011 / End of Life 2027-06-30 / Last Time Buy 2027-01-15 / Replacement: P-0011A",
    "수신: 획득담당 참조\n1. 아래 부품의 생산 중단을 알려 드립니다.\n  가. 품번 : P-0013\n  나. 생산중단일 : 2026.09.30.\n  다. 최종 주문 접수 : 2026.06.30.\n2. 대체 권고품 : 협의 필요",
    "부품 P-0006 은(는) 2026-12-31 부로 단종되며 최종 주문은 2026-10-31 까지 받습니다. 대체품 P-0006-R 검토 바랍니다.",
    "단종공고 / 대상:P-0020 / 단종일:27.02.28 / LTB:26.11.30 / 대체:없음",
]

# ---- 정규화 규칙(서식이 바뀌면 이 사전만 고치면 된다) ----
COL_ALIAS = {"vendor": ["업체명", "공급업체", "vendor"], "country": ["국가", "국적", "nation"], "part_no": ["공급부품", "품목번호", "part"],
             "lead": ["리드타임", "납기(일)", "leadTime"], "last_trade": ["최근거래일", "최종거래", "lastTrade"]}
COUNTRY_MAP = {"국내": "국내", "KR": "국내", "대한민국": "국내", "A국": "가상국가A", "A": "가상국가A", "가상국가A": "가상국가A",
               "B국": "가상국가B", "B": "가상국가B", "가상국가B": "가상국가B", "C국": "가상국가C", "C": "가상국가C", "가상국가C": "가상국가C"}
UNIT_DAYS = {"일": 1, "day": 1, "주": 7, "week": 7, "개월": 30, "month": 30, "년": 365, "year": 365}


def norm_vendor(s):
    # [TODO : 학생 실습 2] 국내외 공급업체 데이터 정제, 부품 단종(DMSMS) 공고 파싱 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def norm_lead(v):
    # [TODO : 학생 실습 2] 국내외 공급업체 데이터 정제, 부품 단종(DMSMS) 공고 파싱 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def norm_date(s):
    s = str(s).strip().rstrip(".")
    m = re.match(r"^(\d{2,4})[.\-/년\s]*(\d{1,2})[.\-/월\s]*(\d{1,2})일?$", s) or re.match(r"^(\d{4})(\d{2})(\d{2})$", s)
    if not m:
        raise ValueError(f"날짜 해석 불가: {s}")
    y, mo, d = (int(x) for x in m.groups())
    return date(y + 2000 if y < 100 else y, mo, d).isoformat()


def pick(row, key):
    for k in COL_ALIAS[key]:
        if k in row:
            return row[k]
    raise KeyError(f"필수 열 없음: {key}")


def to_std(row, src):
    return {"vendor": norm_vendor(pick(row, "vendor")), "country": COUNTRY_MAP[str(pick(row, "country")).strip()],
            "part_no": str(pick(row, "part_no")).strip(), "lead_days": norm_lead(pick(row, "lead")),
            "last_trade": norm_date(pick(row, "last_trade")), "source": src}


def load(text, fmt):
    """형식별 로더: csv / tsv(엑셀 내보내기) / json. 행마다 성공·실패를 따로 센다."""
    if fmt == "json":
        rows = [{"vendor": r["vendor"]["name"], "nation": r["vendor"]["nation"], "part": r["part"], "leadTime": r["leadTime"],
                 "lastTrade": r["lastTrade"]} for r in json.loads(text)["items"]]
    else:
        rows = list(csv.DictReader(io.StringIO(text), delimiter="\t" if fmt == "tsv" else ","))
    ok, fail = [], []
    for r in rows:
        try:
            ok.append(to_std(r, fmt))
        except (KeyError, ValueError) as e:
            fail.append((r, str(e)))
    return ok, fail


def parse_notice(text):
    """서식이 다른 단종 공고에서 부품번호·단종일·LTB·대체품을 추출한다."""
    part = re.search(r"P-\d{4}", text)
    dt = r"(\d{2,4}[년.\-/]\s*\d{1,2}[월.\-/]\s*\d{1,2}일?\.?)"
    eol = re.search(r"(?:단종(?: 예정)?일|생산중단일|End of Life)\s*[:：]?\s*" + dt, text) or re.search(dt + r"\s*부로\s*단종", text)
    ltb = re.search(r"(?:LTB\)?\s*마감|Last Time Buy|최종 주문 접수|최종 주문은|LTB)\s*[:：]?\s*" + dt, text)
    rep = re.search(r"(?:대체품|대체 권고품|Replacement|대체)\s*[:：]?\s*(P-[\w-]+|없음|협의 필요)", text)
    if not (part and eol and ltb):
        raise ValueError("필수 항목(부품번호·단종일·LTB) 추출 실패")
    return {"part_no": part.group(0), "eol_date": norm_date(eol.group(1)), "ltb_date": norm_date(ltb.group(1)),
            "replacement": rep.group(1) if rep else "미기재"}


# 1) 형식별 파싱
results, total, success = [], 0, 0
for fmt, text in [("csv", RAW_CSV), ("tsv", RAW_TSV), ("json", RAW_JSON)]:
    ok, fail = load(text, fmt)
    results += ok; total += len(ok) + len(fail); success += len(ok)
    print(f"[{fmt.upper():4s}] {len(ok) + len(fail)}행 중 {len(ok)}행 성공" + (f", 실패 {[e for _, e in fail]}" if fail else ""))
notices, n_fail = [], 0
for t in NOTICES:
    try:
        notices.append(parse_notice(t))
    except ValueError as e:
        n_fail += 1; print("공고 파싱 실패:", e)
total += len(NOTICES); success += len(notices)
print(f"[공문 텍스트] 단종 공고 {len(NOTICES)}건 중 {len(notices)}건 성공")
rate = success / total
print(f"\n전체 파싱 성공률 {rate:.0%} ({success}/{total}) - 목표 100%: {'달성' if rate == 1 else '미달'}")

# 2) 정제 결과(업체명 표기 통일, 중복 제거)
std = pd.DataFrame(results)
before = len(std)
std = std.drop_duplicates(["vendor", "part_no"]).sort_values(["part_no", "vendor"])
print(f"\n[정제된 공급업체 자료] {before}행 -> 중복 제거 후 {len(std)}행, 업체 {std['vendor'].nunique()}곳")
print(std.to_string(index=False))
print("\n[단종 공고 파싱 결과]")
print(pd.DataFrame(notices).to_string(index=False))

# 3) 서식 변경 대응 시험: 열 이름이 바뀐 새 양식 -> 별칭 사전 한 줄 추가로 처리
new_form = "협력업체,소재국,부품,조달기간,거래일\n가상정밀(주),국내,P-0021,30일,2026.04.01\n"
ok, fail = load(new_form, "csv")
print(f"\n새 양식(열 이름 변경) 1차 시도: 성공 {len(ok)} / 실패 {len(fail)} -> {fail[0][1] if fail else '-'}")
for k, alias in {"vendor": "협력업체", "country": "소재국", "part_no": "부품", "lead": "조달기간", "last_trade": "거래일"}.items():
    COL_ALIAS[k].append(alias)
ok, fail = load(new_form, "csv")
print(f"별칭 사전 갱신 후: 성공 {len(ok)} / 실패 {len(fail)} (코드 수정 없이 설정만 변경)")


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[CSV ] 6행 중 6행 성공
[TSV ] 5행 중 5행 성공
[JSON] 3행 중 3행 성공
[공문 텍스트] 단종 공고 5건 중 5건 성공

전체 파싱 성공률 100% (19/19) - 목표 100%: 달성

[정제된 공급업체 자료] 14행 -> 중복 제거 후 13행, 업체 9곳
       vendor country part_no  lead_days last_trade source
         가상정밀      국내  P-0001         45 2026-03-02    csv
         가상정밀      국내  P-0002         56 2026-03-20   json
VIRTUAL PARTS   가상국가A  P-0003        180 2026-01-10    csv
        가상유공압      국내  P-0003         60 2026-02-27    csv
         가상전자      국내  P-0004         60 2026-03-15    csv
SAMPLE OPTICS   가상국가B  P-0005        210 2026-01-05    csv
         가상광학      국내  P-0005         75 2026-02-11    tsv
 MODEL SENSOR   가상국가C  P-0006        365 2025-11-03    tsv
         가상테크      국내  P-0008         90 2026-03-05    tsv
VIRTUAL PARTS   가상국가A  P-0011        180 2025-12-30    csv
SAMPLE OPTICS   가상국가B  P-0013        200 2026-01-20    tsv
 MODEL SENSOR   가상국가C  P-0014        150 2026-01-09   json
        가상시스템      국내  P-0016         90 2026-02-01   json

[단종 공고 파싱 결과]
part_no   eol_date   ltb_date replacement
 P-0003 2027-03-31 2026-12-31          없음
 P-0011 2027-06-30 2027-01-15     P-0011A
 P-0013 2026-09-30 2026-06-30       협의 필요
 P-0006 2026-12-31 2026-10-31    P-0006-R
 P-0020 2027-02-28 2026-11-30          없음

새 양식(열 이름 변경) 1차 시도: 성공 0 / 실패 1 -> '필수 열 없음: vendor'
별칭 사전 갱신 후: 성공 1 / 실패 0 (코드 수정 없이 설정만 변경)
```
</details>
